# 02 - Statistical analysis
Reads `analysis_df.csv` from `01-merge-and-prep`. Needs `statsmodels` (`python -m uv add statsmodels`).
Unit of analysis is the comment. Standard errors are clustered on `video_id` when it exists.
Caveat: codes come from prompt v1.4 on a 4-bit model, so all results describe the LLM coding, pending the hand-coded check at the end.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

CRISIS = "ateneo-drowning"

# Define both the base output folder and the crisis-specific folder
BASE_OUT = Path("04-statistical-analysis/output")
OUT = BASE_OUT / CRISIS

PHASES = ["01-onset", "02-confirmation", "03-spread", "04-reckoning", "05-lingering"]
EMO = ["NEG_EMOTION", "ANY_EMOTION", "BROAD_OUTRAGE", "ANGER", "CONTEMPT", "DISGUST",
       "SADNESS", "CONDOLENCE", "COMPASSION_VICTIMS", "SYMPATHY_OFFENDER", "FEAR"]

# 01-merge-and-prep writes to the crisis folder (OUT)
df = pd.read_csv(OUT / "analysis_df.csv")

df["phase"] = pd.Categorical(df.phase, PHASES, ordered=True)
top = df.LANGUAGE.value_counts().index[:3]
df["lang3"] = df.LANGUAGE.where(df.LANGUAGE.isin(top), "OTHER")
group_col = df["video_id"] if "video_id" in df else df["comment_id"]
df["cluster"] = group_col.factorize()[0]
print("clustering on:", "video_id" if "video_id" in df else "comment_id (no clustering)")
df.shape

In [ ]:
def fit(formula, d):
    return smf.logit(formula, d).fit(disp=0, cov_type="cluster", cov_kwds={"groups": d.cluster})

def or_row(m, term):
    ci = m.conf_int().loc[term]
    return pd.Series({"OR": np.exp(m.params[term]), "lo": np.exp(ci[0]),
                      "hi": np.exp(ci[1]), "p": m.pvalues[term]})

def ca_trend(x, n, s):
    x, n, s = map(np.asarray, (x, n, s))
    N = n.sum()
    p = x.sum() / N
    t = (s * (x - n * p)).sum()
    v = p * (1 - p) * ((n * s**2).sum() - (n * s).sum()**2 / N)
    z = t / np.sqrt(v)
    return z, 2 * stats.norm.sf(abs(z))

## RQ1 - How much class talk, and does it change across phases?
To determine the baseline prevalence of class-framed discourse and evaluate whether its volume fluctuates across the five crisis phases.

**Statistical Methods:** 
* **Wilson Score Intervals:** To calculate the 95% confidence intervals for the proportion of class talk in each phase.
* **Chi-Square Test of Independence:** To test for an overall relationship between crisis phase and the presence of a class frame[cite: 2].
* **Cochran-Armitage Trend Test:** To determine if there is a linear directional trend (increasing or decreasing) in class talk over time[cite: 2].

**Hypotheses:**
* **Null Hypothesis (H0):** The proportion of class-framed comments remains stable and independent of the crisis phase.
* **Alternative Hypothesis (Ha):** The proportion of class-framed comments changes significantly, showing a distinct trend as the crisis evolves.

In [ ]:
g = df.groupby("phase", observed=True).CLASS_FRAME.agg(["sum", "count"])
lo, hi = proportion_confint(g["sum"], g["count"], method="wilson")
g["share"] = g["sum"] / g["count"]
g["ci_lo"], g["ci_hi"] = lo, hi
g.round(3)

In [ ]:
# 1. Reset the index so 'phase' becomes a standard column
rq1_table = g.reset_index().copy()

# 2. Combine the 'ci_lo' and 'ci_hi' columns into a single formatted 95% CI string
rq1_table["95% CI (ci_lo, ci_hi)"] = "[" + rq1_table["ci_lo"].round(3).astype(str) + ", " + rq1_table["ci_hi"].round(3).astype(str) + "]"

# 3. Format the share as a percentage string for easier reading in a paper (optional)
rq1_table["Class Talk Share"] = (rq1_table["share"] * 100).round(1).astype(str) + "%"

# 4. Rename the existing columns to match your academic headers
rq1_table = rq1_table.rename(columns={
    "phase": "Crisis Phase",
    "sum": "Class Talk Volume",
    "count": "Total Comments"
})

# 5. Select only the specific columns you want to show in the final table
final_rq1_table = rq1_table[[
    "Crisis Phase", 
    "Total Comments", 
    "Class Talk Volume", 
    "Class Talk Share", 
    "95% CI (ci_lo, ci_hi)"
]]

# 6. Display the cleanly formatted table
display(final_rq1_table)

# Optional: Export to HTML to copy-paste into your manuscript
final_rq1_table.to_html("rq1_volume_table.html", index=False)

In [ ]:
tab = pd.crosstab(df.phase, df.CLASS_FRAME)
chi2, p, dof, exp = stats.chi2_contingency(tab)
print(f"chi2({dof}) = {chi2:.2f}, p = {p:.3f}, Cramer's V = {np.sqrt(chi2 / tab.values.sum()):.3f}")

z, pz = ca_trend(g["sum"], g["count"], np.arange(len(g)))
print(f"Cochran-Armitage trend: z = {z:.2f}, p = {pz:.3f}")

rp, cp = tab.sum(1).values[:, None] / tab.values.sum(), tab.sum(0).values / tab.values.sum()
adj = (tab.values - exp) / np.sqrt(exp * (1 - rp) * (1 - cp))
print(pd.DataFrame(adj, index=tab.index, columns=tab.columns).round(2))

m = fit("CLASS_FRAME ~ phase_num + log_len + C(lang3)", df)
print(or_row(m, "phase_num").round(3))

### Direction of class talk (class-framed comments only)

In [ ]:
cf = df[df.CLASS_FRAME == 1].copy()
cf["dir3"] = cf.CLASS_DIRECTION.replace({"INSTITUTION_ELITE": "ELITE_OR_OTHER", "OTHER": "ELITE_OR_OTHER"})
print(cf.dir3.value_counts(), "\n")
print(pd.crosstab(cf.phase, cf.dir3))
print(pd.crosstab(cf.phase, cf.dir3, normalize="index").round(2))

In [ ]:
def perm_chi2(a, b, n=3000, seed=0):
    rng = np.random.default_rng(seed)
    a, b = pd.factorize(a)[0], pd.factorize(b)[0]
    ka, kb = a.max() + 1, b.max() + 1
    def stat(bb):
        t = np.zeros((ka, kb))
        np.add.at(t, (a, bb), 1)
        return stats.chi2_contingency(t, correction=False)[0]
    obs = stat(b)
    sims = np.array([stat(rng.permutation(b)) for _ in range(n)])
    return obs, (1 + (sims >= obs).sum()) / (n + 1)

obs, p = perm_chi2(cf.phase.astype(str), cf.dir3)
print(f"phase x direction: chi2 = {obs:.2f}, permutation p = {p:.3f}")

## RQ2 & RQ3: The Emotion Profile of Class-Framed Discourse
**Objective:** To determine how class talk functions socially by comparing its emotional signature to standard crisis communication. 

**Statistical Methods:**
* **Logistic Regression:** Used to calculate the Odds Ratio (OR) for specific emotions occurring in class-framed comments, controlling for comment length (`log_len`) and language (`C(lang3)`)[cite: 3].
* **Holm-Bonferroni Method:** Applied to correct p-values for multiple comparisons across 11 distinct emotional outcomes, minimizing false positives.

**Hypotheses:**
* **Null Hypothesis (H0):** Class-framed comments are just as likely to contain specific emotions (e.g., sadness, anger) as non-class-framed comments.
* **Alternative Hypothesis (Ha):** Class-framed comments possess a distinctly different emotional profile compared to standard comments.

In [ ]:
def emotion_table(d):
    rows = []
    for y in EMO:
        r = {"outcome": y,
             "rate_class": d.loc[d.CLASS_FRAME == 1, y].mean(),
             "rate_other": d.loc[d.CLASS_FRAME == 0, y].mean()}
        try:
            m = fit(f"{y} ~ CLASS_FRAME + C(phase) + C(lang3) + log_len", d)
            r.update(or_row(m, "CLASS_FRAME"))
        except Exception as e:
            r["note"] = type(e).__name__
        rows.append(r)
    t = pd.DataFrame(rows).set_index("outcome")
    ok = t.p.notna()
    t.loc[ok, "p_holm"] = multipletests(t.loc[ok, "p"], method="holm")[1]
    return t

emo_tab = emotion_table(df)
emo_tab.round(3)

In [ ]:
emo_tab.to_csv(OUT / "emotion_by_class.csv")

Summary table

In [ ]:
# 1. Create a copy and reset the index so 'outcome' becomes a standard column
clean_table = emo_tab.reset_index().copy()

# 2. Combine the 'lo' and 'hi' columns into a single formatted 95% CI string column
clean_table["95% CI (lo, hi)"] = "[" + clean_table["lo"].round(3).astype(str) + ", " + clean_table["hi"].round(3).astype(str) + "]"

# 3. Rename the existing columns to match your academic headers
clean_table = clean_table.rename(columns={
    "outcome": "Emotional Outcome",
    "rate_class": "Class Talk Rate",
    "rate_other": "Other Comments Rate",
    "OR": "Odds Ratio (OR)",
    "p_holm": "Adj. p-value (Holm)"
})

# 4. Select only the specific columns you want to show in the final table
final_table = clean_table[[
    "Emotional Outcome", 
    "Class Talk Rate", 
    "Other Comments Rate", 
    "Odds Ratio (OR)", 
    "95% CI (lo, hi)", 
    "Adj. p-value (Holm)"
]].round(3) # Round the remaining numeric columns to 3 decimal places

# 5. Display the beautifully formatted table
display(final_table)

### Within class talk: does the direction of class talk carry different emotions?

In [ ]:
two = cf[cf.dir3.isin(["INEQUALITY_JUSTICE", "VICTIMS_POOR"])]
rows = []
for y in EMO:
    a = two.loc[two.dir3 == "INEQUALITY_JUSTICE", y]
    b = two.loc[two.dir3 == "VICTIMS_POOR", y]
    odds, p = stats.fisher_exact([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])
    rows.append({"outcome": y, "rate_ineq_justice": a.mean(), "rate_victims_poor": b.mean(), "OR": odds, "p": p})
within = pd.DataFrame(rows).set_index("outcome")
within["p_holm"] = multipletests(within.p, method="holm")[1]
within.round(3)

## Phase and Class Interaction
**Objective:** To test the "emotional turn" argument by evaluating if the specific emotions attached to class talk shift as the crisis matures.

**Statistical Methods:**
* **Likelihood-Ratio (LR) Test:** Compares a baseline logistic model against an interaction model (`CLASS_FRAME * phase_num`) to see if the passage of time significantly alters the emotional weight of class talk.

**Hypotheses:**
* **Null Hypothesis (H0):** The emotional profile of class talk is static; the crisis phase does not interact with the class frame.
* **Alternative Hypothesis (Ha):** There is a significant interaction effect between class framing and phase, meaning the emotions embedded in class talk evolve over the crisis lifecycle.

In [ ]:
rows = []
for y in ["NEG_EMOTION", "ANGER", "CONTEMPT", "SADNESS", "COMPASSION_VICTIMS", "BROAD_OUTRAGE"]:
    m0 = smf.logit(f"{y} ~ CLASS_FRAME + phase_num + C(lang3) + log_len", df).fit(disp=0)
    m1 = smf.logit(f"{y} ~ CLASS_FRAME * phase_num + C(lang3) + log_len", df).fit(disp=0)
    lr = 2 * (m1.llf - m0.llf)
    rows.append({"outcome": y, "LR": lr, "p": stats.chi2.sf(lr, 1),
                 "interaction_OR": np.exp(m1.params["CLASS_FRAME:phase_num"])})
inter = pd.DataFrame(rows).set_index("outcome")
inter["p_holm"] = multipletests(inter.p, method="holm")[1]
inter.round(3)

## Robustness - rerun the class vs emotion models on stricter subsets

In [ ]:
df.autofix.value_counts(dropna=False)

In [ ]:
subsets = {"all": df}
if "confidence" in df:
    subsets["conf HIGH/MEDIUM"] = df[df.confidence.astype(str).str.upper().isin(["HIGH", "MEDIUM"])]
if "autofix" in df:
    subsets["autofix False"] = df[df.autofix.astype(str).str.lower().isin(["false", "0"])]
subsets["no NEEDS_CONTEXT"] = df[df.NEEDS_CONTEXT == 0]

for k, d in subsets.items():
    print(k, len(d), "class-framed:", int(d.CLASS_FRAME.sum()))

subsets = {k: d for k, d in subsets.items() if d.CLASS_FRAME.sum() > 0}
rob = pd.concat({k: emotion_table(d)[["OR", "p_holm"]] for k, d in subsets.items()}, axis=1)
rob.round(3)

## Qualitative anchors - random class-framed comments per phase and direction

In [ ]:
pd.set_option("display.max_colwidth", 300)
ex = (cf.sample(frac=1, random_state=1)
        .groupby(["phase", "dir3"], observed=True)
        .head(2)
        .sort_values(["phase", "dir3"]))
ex[["phase", "dir3", "comment_id", "text"]]

## Validation sample for hand coding (blind) - use for Cohen's kappa

In [ ]:
rng = 42
pos = df[df.CLASS_FRAME == 1].sample(75, random_state=rng)
neg = df[df.CLASS_FRAME == 0].sample(75, random_state=rng)
samp = pd.concat([pos, neg]).sample(frac=1, random_state=rng)

blind = samp[["comment_id", "phase", "text"]].copy()
for c in ["hand_CLASS_FRAME", "hand_CLASS_DIRECTION", "hand_EMOTIONS"]:
    blind[c] = ""
blind.to_csv(OUT / "validation_sample_blind.csv", index=False)
samp[["comment_id", "CLASS_FRAME", "CLASS_DIRECTION"] + EMO[3:]].to_csv(OUT / "validation_sample_llm_key.csv", index=False)
print(len(blind))

After hand coding, compute agreement:
`from sklearn.metrics import cohen_kappa_score` then `cohen_kappa_score(hand, llm)` per variable.
The sample is stratified (half class-framed), so also report precision and recall for CLASS_FRAME, not kappa alone.